In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from tqdm.notebook import tqdm
import time
import sys
import os

# Ensure the notebook can find your src/ folder
sys.path.append(os.path.abspath('..'))
from src.core import RAGGuardPipeline

# Initialize the pipeline
pipeline = RAGGuardPipeline()
print("Pipeline loaded successfully for benchmarking!")


In [ ]:
print("Downloading HaluEval benchmark dataset...")
# Load the QA subset of HaluEval (takes a few seconds)
dataset = load_dataset("UCL-DarkLab/HaluEval", "qa", split="train[:1000]")

test_data = []
# Create a balanced dataset: Even indices = Faithful, Odd indices = Hallucination
for i, row in enumerate(dataset):
    if i % 2 == 0:
        # Ground Truth: 0 (Faithful / No Hallucination)
        test_data.append({
            "context":    row['knowledge'],
            "response":   row['right_answer'],
            "true_label": 0
        })
    else:
        # Ground Truth: 1 (Hallucination present)
        test_data.append({
            "context":    row['knowledge'],
            "response":   row['hallucinated_answer'],
            "true_label": 1
        })

df = pd.DataFrame(test_data)
print(f"Dataset prepared! Total samples: {len(df)} "
      f"({(df['true_label']==0).sum()} Faithful, {(df['true_label']==1).sum()} Hallucinated)")
df.head()


In [ ]:
predictions = []
execution_times = []

print("Running pipeline against ground-truth dataset...")
for idx, row in tqdm(df.iterrows(), total=len(df)):
    try:
        t0 = time.perf_counter()
        # pipeline.evaluate() returns a plain list of claim-result dicts
        report = pipeline.evaluate(
            generated_text=row['response'],
            source_text=row['context']
        )
        elapsed = time.perf_counter() - t0

        # Binary classification: any Contradiction → Hallucinated (1)
        is_hallucination = any(claim['nli_label'] == 'Contradiction' for claim in report)
        predictions.append(1 if is_hallucination else 0)
        execution_times.append(elapsed)
    except Exception as e:
        # Fallback for empty strings or unexpected data
        predictions.append(0)
        execution_times.append(0)

df['predicted_label'] = predictions
print(f"\nEvaluation complete!")
print(f"Average latency per request : {np.mean(execution_times)*1000:.2f} ms")
print(f"Total evaluation time       : {sum(execution_times):.1f} s")


In [ ]:
accuracy  = accuracy_score( df['true_label'], df['predicted_label'])
precision = precision_score(df['true_label'], df['predicted_label'])
recall    = recall_score(   df['true_label'], df['predicted_label'])
f1        = f1_score(       df['true_label'], df['predicted_label'])

print("─" * 55)
print("  RAG-Guard NLI  ·  Performance Metrics")
print("─" * 55)
print(f"  Accuracy  : {accuracy:.4f}   (overall correctness)")
print(f"  Precision : {precision:.4f}   (when it flags, how often is it right?)")
print(f"  Recall    : {recall:.4f}   (what fraction of hallucinations did it catch?)")
print(f"  F1-Score  : {f1:.4f}   (harmonic mean of precision & recall)")
print("─" * 55)


In [ ]:
cm = confusion_matrix(df['true_label'], df['predicted_label'])

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Faithful (0)', 'Hallucination (1)'],
    yticklabels=['Faithful (0)', 'Hallucination (1)'],
    annot_kws={"size": 16}
)
plt.title('RAG-Guard NLI vs. Human Annotations (HaluEval)', fontsize=14, pad=15)
plt.ylabel('True Human Label', fontsize=12)
plt.xlabel('RAG-Guard Predicted Label', fontsize=12)
plt.tight_layout()

out_path = os.path.join(os.getcwd(), 'benchmark_results.png')
plt.savefig(out_path, dpi=300)
plt.show()
print(f"Confusion matrix saved → {out_path}")
